# AksharDrishti — Kaggle / Colab setupRun this top to bottom on a **GPU** runtime. It installs deps, pulls the datasets, and launches training.**Before you start:** accept the IndicDLP terms at https://huggingface.co/datasets/ai4bharat/indicdlp

## 1. Environment check

In [ ]:
import subprocess, sysprint(sys.version)try:    print(subprocess.check_output(["nvidia-smi"]).decode())except Exception as e:    print("NO GPU DETECTED —", e)    print("Kaggle: Settings → Accelerator → GPU T4 x2.  Account must be phone-verified.")

## 2. Install

In [ ]:
!pip install -q ultralytics sahi transformers datasets huggingface-hub jiwer wandb pymupdf streamlit!apt-get install -qq -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar fonts-indic fonts-noto-core > /dev/null!pip install -q pytesseractprint("done")

## 3. Get the code onto the machinePick **one** of the two options below.### Option A — Kaggle Dataset (fastest, no GitHub needed)1. Kaggle → **Datasets** → **New Dataset**2. Upload `AksharDrishti.zip` (Kaggle unzips it automatically)3. Title it `akshardrishti-code`, set visibility to **Private**, click Create4. Back in this notebook: **Add Input** (top right) → **Datasets** → your datasetColab equivalent: upload the zip to Drive, mount it, and `!unzip -q /content/drive/MyDrive/AksharDrishti.zip`.### Option B — GitHub (better once more than one of you is editing code)On any machine with the unzipped folder:```bashcd aksharDrishtigit init && git add . && git commit -m "AksharDrishti: initial pipeline"gh repo create akshardrishti --private --source=. --push# no gh CLI? create the repo on github.com, then:#   git remote add origin https://github.com/<you>/akshardrishti.git#   git branch -M main && git push -u origin main```Then set `REPO` below. **Use a private repo** — AksharDrishti-Bench will contain governmentdocuments, and `.gitignore` already excludes `benchmark/images/` for that reason.With four people committing over two weeks, Option B pays for itself by about day 3.

In [ ]:
REPO = ""   # Option B: paste your git URL here. Leave "" to use Option A.import os, sys, glob, shutiltarget = "/kaggle/working/aksharDrishti" if os.path.isdir("/kaggle/working") else os.path.abspath("aksharDrishti")if REPO:    if not os.path.isdir(target):        !git clone -q $REPO $targetelse:    # Find the code inside whatever the Kaggle Dataset got named.    hits = [d for d in glob.glob("/kaggle/input/*/**/akshardrishti", recursive=True) if os.path.isdir(d)]    hits += [d for d in glob.glob("/kaggle/input/*/**/configs", recursive=True) if os.path.isdir(d)]    if not hits:        raise SystemExit(            "Code not found under /kaggle/input.\n"            "Did you add the dataset as an Input (top-right panel)?\n"            "Contents of /kaggle/input: " + str(glob.glob("/kaggle/input/*"))        )    src = os.path.dirname(sorted(hits, key=len)[0])   # parent of akshardrishti/ or configs/    if not os.path.isdir(target):        shutil.copytree(src, target)                  # /kaggle/input is READ-ONLY; we must copy    print("copied from:", src)os.chdir(target)sys.path.insert(0, target)print("cwd:", os.getcwd())assert os.path.isfile("configs/pipeline.yaml"), "wrong folder - configs/pipeline.yaml not found"print("files:", sorted(os.listdir(".")))

## 4. Verify the install58 unit tests, no model weights required. If these fail, stop and fix before training.

In [ ]:
!python tests/run_tests_nopytest.py 2>&1 | tail -6

## 5. Hugging Face login (for IndicDLP)

In [ ]:
from huggingface_hub import loginlogin()   # paste a token with 'read' scope

## 6. Prepare the IndicDLP subset**Inspect first.** Our 42-class map was reconstructed from the paper; if any real class name differs,the export stops rather than silently dropping annotations. Reconcile anything it flags into`configs/class_map.yaml`, then export.

In [ ]:
!python data/prepare_indicdlp.py --inspect --inspect-limit 2000

In [ ]:
!python data/prepare_indicdlp.py --out datasets/indicdlp_subset --max-images 15000!cat datasets/indicdlp_subset/subset_stats.json

## 7. Train the layout model (model #1)~10–14h on 2×T4 for 60 epochs. Launch it overnight. Checkpoints save every epoch.

In [ ]:
BACKUP = "/kaggle/working/backup"   # or your mounted Drive path!python train/train_layout.py \    --data datasets/indicdlp_subset/data.yaml \    --model yolo11l.pt \    --epochs 60 --batch 16 --imgsz 1024 --device 0,1 \    --backup-dir $BACKUP

## 8. Baseline comparisonEvaluate a released IndicDLP checkpoint without training — this is the baseline row in your layout table.

In [ ]:
# download a released IndicDLP checkpoint first, then:# !python train/train_layout.py --eval-only --weights weights/indicdlp_yolov10x.pt \#     --data datasets/indicdlp_subset/data.yaml

## 9. Prepare Mozhi and train the recogniser (model #2)Run this on a **different team member's account**, in parallel with the layout training above —that is the whole point of having four Kaggle accounts.

In [ ]:
!python data/prepare_mozhi.py --languages hindi marathi --out datasets/mozhi

In [ ]:
!python train/train_crnn.py --data datasets/mozhi --epochs 30 --batch 64 \    --backup-dir /kaggle/working/backup

## 10. Run the benchmarkOnce AksharDrishti-Bench is annotated and both models are trained.

In [ ]:
!python eval/run_benchmark.py --benchmark benchmark --out results/ --ablations!cat results/results.md

## 11. Save your work**Kaggle output is lost if the kernel is killed.** Copy weights out before the session ends.

In [ ]:
!mkdir -p /kaggle/working/final!cp -v runs/detect/*/weights/best.pt /kaggle/working/final/ 2>/dev/null || true!cp -v weights/recognize/*.pt weights/recognize/charset_deva.txt /kaggle/working/final/ 2>/dev/null || true!ls -lh /kaggle/working/final/